In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import time
import joblib
import pandas as pd

sys.path.append('../src')

## Чтение данных

In [3]:
data = joblib.load('data/data_mmlu.pkl')

In [4]:
data

,query,answer,true_answer
0,\nКонтекст:\nЧто такое ипотека и ипотечный кре...,A) Жилое помещение,['D']
1,\nКонтекст:\nКредитные карты.\n\nКакие действи...,D) 60 месяцев,['A']
2,\nКонтекст:\nПриобретение готового жилья\nГото...,C) Отсутствие возможности оформления постоянно...,['C']
3,\nКонтекст:\nПриобретение строящегося жилья.\n...,[C] 30%,['C']
4,\nКонтекст:\nКоммерческая недвижимость\n\nУсло...,[A] Возраст заемщика от 21 до 75 лет.,['A']


## Создание Sampler'а и Scorer'а

In [5]:
from llm_val.sampler.MMLUSampler import MMLUSampler

sampler = MMLUSampler(data)

In [6]:
from llm_val.scorer.UniversalScorer import UniversalScorer
from llm_val.utils.metrics import METRICS

In [7]:
scorer = UniversalScorer(metrics=METRICS)

In [8]:
METRICS

{'single_mean': {'call': <function llm_val.utils.metrics.<lambda>(x)>,
  'is_singlecol': True},
 'multicol_mean': {'call': <function llm_val.utils.metrics.<lambda>(x)>,
  'is_singlecol': False}}

## Использование валидационных тестов

Все валидационные тесты должны находиться в директории `val_test` и быть распределены по директориям в зависимости от блока тестов. Например, тест на ключевую метрику должен лежать в директории `llm_val/val_test/model_quality`.

Каждый тест должен возвращать словарь следующего вида: `{'report': dict, 'precomputed': dict}`, где:
- `precomputed` - это словарь с данными, которые валидатор хочет сохранить;
- `report` - словарь следующего вида: `{'semaphore': str, 'result_plots': list, 'result_dataframes': list}`, где:
    - `semaphore` - цвет светофора. Одно из четырех значений: `['green', 'yellow', 'red', 'gray']`;
    - `result_plots` - список с графиками для отчета;
    - `result_dataframes` - список с DataFrame(таблицами) для отчета.

В каждом тесте, где есть светофор, должен быть параметр `is_info`, чтобы иметь возможность сделать тест информативным ("отключить" светофор).

В конце параметров каждого теста должен быть `**kwargs`.

Пример теста смотрите в `llm_val/val_test/model_quality/valtest_metric.py`.

In [9]:
from llm_val.val_test.model_quality.valtest_metric import valtest_metric

### Запуск теста

In [10]:
res = valtest_metric(sampler=sampler,
                     scorer=scorer,
                     main_metric='single_mean_metric_value',
                     data_type='test',
                     metric_name='single_mean',
                     semaphore_threshold={'default': (0.2, 0.4)},
                     greater_is_better={'default': True},
                     is_info=False,
                     metric_value=None)

In [11]:
print(res['report'])

{'semaphore': 'green', 'result_plots': [], 'result_dataframes': [   Значение ключевой метрики "single_mean_metric_value" на test  \
0                                                0.6              

  Результат теста  
0           green  ]}


In [12]:
res['report']['result_dataframes'][0]

,"Значение ключевой метрики ""single_mean_metric_value"" на test",Результат теста
0,0.6,green


In [13]:
res['precomputed']

{'metric_value': {'single_mean_metric_value': 0.6}}

### Запуск с использованием предподсчитанных значений

Каждый тест должен иметь возможность запускаться с предподсчитанными значениями(функционал `precomputed`). Для этого необходимо в параметрах функции указать значения, которые сохраняются в `precomputed`. Название ключа в `precomputed` и параметрах функции должны совпадать! См. пример в `llm_val/val_test/model_quality/valtest_metric.py`

In [14]:
res = valtest_metric(sampler=sampler,
                     scorer=scorer,
                     main_metric='mean',
                     data_type='test',
                     metric_name='single_mean',
                     semaphore_threshold={'default': (0.2, 0.4)},
                     greater_is_better={'default': True},
                     is_info=False,
                     metric_value={'mean': 0.32})

In [15]:
print(res['report'])

{'semaphore': 'red', 'result_plots': [], 'result_dataframes': [   Значение ключевой метрики "mean" на test Результат теста
0                                      0.32             red]}


In [16]:
res['report']['result_dataframes'][0]

,"Значение ключевой метрики ""mean"" на test",Результат теста
0,0.32,red


In [17]:
res['precomputed']

{'metric_value': {'mean': 0.32}}

### Целевой запуск(как тесты запускаются в validation)

Ниже представлена реализация того, как тесты запускаются в агрегирующей функции(validation) (Именно поэтому важно в конце парметров каждого теста писать `**kwargs`).

In [18]:
test_params = {'sampler': sampler,
               'scorer': scorer,
               'main_metric':'single_mean_metric_value',
               'data_type': 'test', 
               'metric_name': None, 
               'semaphore_threshold': {'default': (0.2, 0.4)},
               'greater_is_better': {'default': True},
               'is_info': False, 
               'metric_value': None}

In [19]:
res = valtest_metric(**test_params)

In [20]:
print(res['report'])

{'semaphore': 'green', 'result_plots': [], 'result_dataframes': [   Значение ключевой метрики "single_mean_metric_value" на test  \
0                                                0.6              

  Результат теста  
0           green  ,               Значение метрики на test Результат теста
multicol_mean                      0.6           green]}


In [21]:
res['report']['result_dataframes'][0]

,"Значение ключевой метрики ""single_mean_metric_value"" на test",Результат теста
0,0.6,green


In [22]:
res['precomputed']

{'metric_value': {'single_mean_metric_value': 0.6, 'multicol_mean': 0.6}}

## Примечание

Мы можем провести этот тест и для MMLU датасета, и для RuBQ датасета, и для LongAnswer датасета. Унификация реализации теста (отсутствие реализации одного и того же теста для каждого вида датасетов) достигается благодаря sampler'у и scorer'у и их стандартизации (подробнее см. в Sampler_example.ipynb и Scorer_example.ipynb).

Вся сложность в запуске теста сводится к правильной реализации Sampler'a и Scorer'а.

In [23]:
from llm_val.utils.metrics import METRICS
from llm_val.sampler.MMLUSampler import MMLUSampler
from llm_val.sampler.RuBQSampler import RuBQSampler
from llm_val.scorer.UniversalScorer import UniversalScorer
from llm_val.sampler.LongAnswerSampler import LongAnswerSampler

/home/user/conda/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [24]:
data_mmlu = joblib.load('data/data_mmlu.pkl')
data_rubq = joblib.load('data/data_rubq.pkl')
data_la = joblib.load('data/data_longanswer.pkl')

In [25]:
sampler_mmlu = MMLUSampler(data_mmlu)
sampler_rubq = RuBQSampler(data_rubq)
sampler_la = LongAnswerSampler(data_la)

In [26]:
scorer = UniversalScorer(metrics=METRICS)

In [27]:
res_mmlu = valtest_metric(sampler=sampler_mmlu, scorer=scorer, main_metric='single_mean_metric_value',)
res_rubq = valtest_metric(sampler=sampler_rubq, scorer=scorer, main_metric='single_mean_metric_value',)
res_la = valtest_metric(sampler=sampler_la, scorer=scorer, main_metric='single_mean_Как вы оцениваете качество CodeReview?')

In [28]:
res_mmlu

{'report': {'semaphore': 'green',
  'result_plots': [],
  'result_dataframes': [   Значение ключевой метрики "single_mean_metric_value" на test  \
   0                                                0.6              
   
     Результат теста  
   0           green  ]},
 'precomputed': {'metric_value': {'single_mean_metric_value': 0.6}}}

In [29]:
res_rubq

{'report': {'semaphore': 'green',
  'result_plots': [],
  'result_dataframes': [   Значение ключевой метрики "single_mean_metric_value" на test  \
   0                                                0.8              
   
     Результат теста  
   0           green  ]},
 'precomputed': {'metric_value': {'single_mean_metric_value': 0.8}}}

In [30]:
res_la

{'report': {'semaphore': 'green',
  'result_plots': [],
  'result_dataframes': [   Значение ключевой метрики "single_mean_Как вы оцениваете качество CodeReview?" на test  \
   0                                                3.4                                        
   
     Результат теста  
   0           green  ,
                                                      Значение метрики на test  \
   single_mean_Вы согласны с замечаниями и проблем...                      2.2   
   single_mean_Все существенные проблемы перечисле...                      3.0   
   single_mean_CodeReview не содержит токсичных ко...                      3.0   
   single_mean_Ответ лаконичен и по существу?                              3.0   
   single_mean_Ответ структурирован, все параметры...                      2.6   
   
                                                      Результат теста  
   single_mean_Вы согласны с замечаниями и проблем...           green  
   single_mean_Все существенные проблемы